# Banking77 LoRA Fine-Tuning — Kaggle Orchestration
Clones the repo, installs deps (leaving Kaggle's preinstalled torch alone per outside-voice E4), and runs the pipeline: data prep -> baseline eval -> LoRA training (4 configs) -> validation-based selection -> one fine-tuned test eval -> report generation.

**Kaggle settings (right panel):** Accelerator = **GPU T4 x2** (not P100: poor 4-bit/bf16 support), Internet = **On** (needs a phone-verified account). Only GPU 0 is used; the 1.5B model fits one T4.

**Run it with Save Version -> Save & Run All (Commit):** runs in the background (12h cap) and saves everything in `/kaggle/working` as the version's output. Checkpoints and eval predictions both live there, so every cell is safe to re-run: finished work is skipped, partial work resumes.

**Multi-run training:** each run trains at most 2 configs (Step 3), so the full pipeline takes 2-3 runs. **To continue:** Add Input -> Your Work -> this notebook's previous version. Cell 3 copies its checkpoints/predictions back into `/kaggle/working` automatically.

In [ ]:
# Always start from a fresh clone of main (a stale `repo/` dir makes `git clone` fail and silently keeps old code).
# `!` shell commands don't stop the notebook on failure, so check explicitly and fail fast with a clear cause.
import os
import shutil
import socket
import subprocess

try:
    socket.gethostbyname('github.com')
except OSError as e:
    raise RuntimeError(
        'No internet: cannot resolve github.com. Turn Internet On in notebook settings '
        '(requires a phone-verified Kaggle account).'
    ) from e

os.chdir('/kaggle/working')
shutil.rmtree('repo', ignore_errors=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/V1cky16112003/banking77-lora-finetuning.git', 'repo'],
    check=True,
)
os.chdir('repo')
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

In [ ]:
# Install deps EXCEPT torch (Kaggle's preinstalled CUDA-matched torch build stays as-is, per E4).
import sys

# Strip comments/blank lines since requirements are passed as args, not via -r.
reqs = [line.split('#')[0].strip() for line in open('requirements.txt')]
reqs = [r for r in reqs if r and not r.startswith('torch')]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *reqs], check=True)

In [ ]:
import os
import shutil
from pathlib import Path

# Hide the second T4 before torch loads: with 2 visible GPUs, HF Trainer wraps the model in
# DataParallel, which doubles the effective batch size and breaks with 4-bit QLoRA.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# /kaggle/working is saved as the notebook's output on commit; that's our Drive replacement.
WORK = Path('/kaggle/working')
CHECKPOINT_ROOT = WORK / 'banking77-lora-checkpoints'
# Predictions persisted too, so evaluate_dataset's incremental JSONL resume survives a session timeout.
EVAL_ROOT = WORK / 'banking77-lora-eval'

# Resume: if a previous version's output is attached as input (read-only), copy it back into working.
for name in ('banking77-lora-checkpoints', 'banking77-lora-eval'):
    dest = WORK / name
    for src in Path('/kaggle/input').glob(f'**/{name}'):
        if src.is_dir() and not dest.exists():
            shutil.copytree(src, dest)
            print(f'Resumed {name} from {src}')

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

## Step 1: Load config and data (design-doc step 1-2)

In [ ]:
from src.config import load_task_config
from src.data import load_banking77_splits

config = load_task_config()
# Raises if configs/task.yaml's 77 labels don't exactly match the loaded dataset.
splits = load_banking77_splits(config)

print(f'{len(splits.labels)} labels, dataset revision: {splits.dataset_revision}')
print(f'train={len(splits.train)} val={len(splits.val)} test={len(splits.test)} (frozen, untouched until final eval)')

## Step 2: Baseline evaluation (zero-shot) on the frozen test set (design-doc step 3)

In [ ]:
import gc
import torch
from transformers import AutoTokenizer
from src.evaluate import evaluate_dataset
from src.evaluate_core import compute_macro_f1
from src.train import MODEL_NAME, load_base_model

def to_examples(ds):
    return [{'text': ex['text'], 'label': splits.labels[ex['label']]} for ex in ds]

def macro_f1(results):
    return compute_macro_f1([t for t, _ in results], [p for _, p in results], config.labels)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
test_examples = to_examples(splits.test)

base_model = load_base_model(use_qlora=False)
base_model.eval()
baseline_results = evaluate_dataset(
    base_model, tokenizer, test_examples, config, EVAL_ROOT / 'baseline_test_predictions.jsonl'
)
print('baseline test macro-F1:', round(macro_f1(baseline_results), 4))

# Free the GPU before training loads its own copy of the model.
del base_model
gc.collect()
torch.cuda.empty_cache()

## Step 3: LoRA fine-tuning -- 4 candidate configs (3 LoRA + 1 QLoRA) (design-doc step 4)
Trains at most `MAX_CONFIGS_PER_RUN` unfinished configs, so each commit run ends well inside Kaggle's 12h limit and its output (checkpoints + predictions) is saved; a cancelled or timed-out run saves nothing. Re-run with the previous version attached as input to train the rest. Progress prints one line per 10 steps.

In [ ]:
from src.train import CANDIDATE_CONFIGS, is_training_complete, train_lora

MAX_CONFIGS_PER_RUN = 2

pending = [rc for rc in CANDIDATE_CONFIGS if not is_training_complete(CHECKPOINT_ROOT / rc.name)]
print('Already trained:', [rc.name for rc in CANDIDATE_CONFIGS if rc not in pending])
for run_config in pending[:MAX_CONFIGS_PER_RUN]:
    train_lora(config, splits, run_config, CHECKPOINT_ROOT / run_config.name)
    gc.collect()
    torch.cuda.empty_cache()

ALL_TRAINED = all(is_training_complete(CHECKPOINT_ROOT / rc.name) for rc in CANDIDATE_CONFIGS)
if not ALL_TRAINED:
    print('Configs still to train:', [rc.name for rc in pending[MAX_CONFIGS_PER_RUN:]])
    print('Stopping here so this run saves its output. Re-run with this version attached as input.')

## Step 4: Evaluate each config on validation, pick the best, confirm once on the frozen test set (design-doc step 5)

In [ ]:
import json
from peft import PeftModel

# Selection needs all 4 configs; skip until a later run finishes training them.
if ALL_TRAINED:

    def load_finetuned(run_config):
        # QLoRA adapters are evaluated on the same 4-bit base they were trained on.
        base = load_base_model(run_config.use_qlora)
        model = PeftModel.from_pretrained(base, str(CHECKPOINT_ROOT / run_config.name))
        model.eval()
        return model

    val_examples = to_examples(splits.val)
    val_scores = {}
    for run_config in CANDIDATE_CONFIGS:
        model = load_finetuned(run_config)
        results = evaluate_dataset(model, tokenizer, val_examples, config, EVAL_ROOT / f'val_{run_config.name}.jsonl')
        val_scores[run_config.name] = macro_f1(results)
        print(f'{run_config.name}: val macro-F1 = {val_scores[run_config.name]:.4f}')
        del model
        gc.collect()
        torch.cuda.empty_cache()

    best = max(CANDIDATE_CONFIGS, key=lambda rc: val_scores[rc.name])
    print('Selected on validation:', best.name)
    (EVAL_ROOT / 'val_scores.json').write_text(json.dumps({'scores': val_scores, 'selected': best.name}, indent=2))

    # The frozen test set is touched exactly once, by the validation-selected config only.
    model = load_finetuned(best)
    finetuned_results = evaluate_dataset(
        model, tokenizer, test_examples, config, EVAL_ROOT / 'finetuned_test_predictions.jsonl'
    )
    print(f'{best.name}: test macro-F1 = {macro_f1(finetuned_results):.4f}')

## Step 5: Generate the report (results table, per-class F1, confusion heatmap, bootstrap CI) -- item 2/3, E1

In [ ]:
# `-m` from the repo root so `import src` resolves.
if ALL_TRAINED:
    !python -m scripts.generate_report {EVAL_ROOT}/baseline_test_predictions.jsonl --output-dir {EVAL_ROOT}/report_baseline
    !python -m scripts.generate_report {EVAL_ROOT}/finetuned_test_predictions.jsonl --output-dir {EVAL_ROOT}/report_finetuned

## Step 6: Download the adapter checkpoint locally for serving (D5)
Download `banking77-lora-checkpoints/<selected config>/` from this version's **Output** tab, then run `ADAPTER_DIR=<that dir> uvicorn src.serve:app` locally.